In [ ]:
# CELL 1: SETUP & IMPORT

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

file_path = "dynamic_supply_chain_logistics_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")

In [ ]:
# CELL 2: DATA OVERVIEW

print("Shape of dataset:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nData types:")
print(df.dtypes)

print("\nDescriptive statistics:")
display(df.describe(include='all'))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicated rows:")
print(df.duplicated().sum())

In [ ]:
# CELL 3: DEFINE COLUMN TYPES

datetime_cols = ['timestamp']

categorical_cols = ['risk_classification']

target_col = 'delivery_time_deviation'

# Detect numerical columns automatically
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target column from numerical feature columns
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

print("Datetime columns:", datetime_cols)
print("Categorical columns:", categorical_cols)
print("Numerical feature columns:", numerical_cols)
print("Target column:", target_col)

In [ ]:
# CELL 4: DATA TYPE CONVERSION

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

df['risk_classification'] = df['risk_classification'].astype('category')

for col in numerical_cols + [target_col]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types after conversion:")
print(df.dtypes)

In [ ]:
# CELL 5: TIME-BASED FEATURE ENGINEERING

df['hour_of_day'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.month
df['year'] = df['timestamp'].dt.year
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

df['is_peak_hour'] = df['hour_of_day'].isin([7, 8, 9, 17, 18, 19]).astype(int)

display(df.head())

In [ ]:
# CELL 6: MISSING VALUE ANALYSIS

missing_report = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percentage': df.isnull().mean() * 100
})

display(missing_report.sort_values(by='missing_percentage', ascending=False))

In [ ]:
# CELL 7: DUPLICATE CHECK

duplicate_count = df.duplicated().sum()

print("Number of duplicated rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicated rows found.")

In [ ]:
# CELL 8: OUTLIER DETECTION USING IQR

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]
    
    return outliers, lower_bound, upper_bound

outlier_summary = []

for col in numerical_cols + [target_col]:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    outlier_summary.append({
        'column': col,
        'outlier_count': len(outliers),
        'lower_bound': lower,
        'upper_bound': upper,
        'min_value': df[col].min(),
        'max_value': df[col].max()
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
display(outlier_summary_df.sort_values(by='outlier_count', ascending=False))

In [ ]:
# CELL 9: BOXPLOT FOR TARGET VARIABLE

plt.figure(figsize=(8, 4))
sns.boxplot(x=df[target_col])
plt.title("Boxplot of Delivery Time Deviation")
plt.xlabel("Delivery Time Deviation")
plt.show()

In [ ]:
# Boxplots for important numerical features

important_cols = [
    'fuel_consumption_rate',
    'traffic_congestion_level',
    'eta_variation_hours',
    'iot_temperature',
    'route_risk_level',
    'delay_probability',
    'delivery_time_deviation'
]

for col in important_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

In [ ]:
# CELL 10: OPTIONAL OUTLIER CAPPING FOR TARGET

outliers, lower, upper = detect_outliers_iqr(df, target_col)

df[target_col] = np.where(df[target_col] > upper, upper, df[target_col])
df[target_col] = np.where(df[target_col] < lower, lower, df[target_col])

print("Target outliers capped using IQR bounds.")

In [ ]:
df['traffic_intensity_proxy'] = (
    df['traffic_congestion_level'] * df['is_peak_hour']
)

In [ ]:
df['route_complexity'] = (
    df['traffic_congestion_level'] +
    df['route_risk_level'] +
    df['weather_condition_severity'] * 10
) / 3

In [ ]:
df['lat_bin'] = pd.cut(df['vehicle_gps_latitude'], bins=5, labels=False)
df['long_bin'] = pd.cut(df['vehicle_gps_longitude'], bins=5, labels=False)

df['gps_zone'] = df['lat_bin'].astype(str) + "_" + df['long_bin'].astype(str)

In [ ]:
zone_density = df['gps_zone'].value_counts().to_dict()

df['delivery_zone_density'] = df['gps_zone'].map(zone_density)

In [ ]:
df['distance_proxy_x_peak_hour'] = df['eta_variation_hours'] * df['is_peak_hour']

df['traffic_x_weather'] = (
    df['traffic_congestion_level'] *
    df['weather_condition_severity']
)

df['traffic_x_route_risk'] = (
    df['traffic_congestion_level'] *
    df['route_risk_level']
)

In [ ]:
# CELL 11: EXPORT CLEANED DATASET

df.to_csv("cleaned_dynamic_supply_chain_logistics_dataset.csv", index=False)

print("Cleaned dataset exported successfully!")